# Cross-sectional stat-arb: does letting stocks see each other help?

Baseline is the CNN+Transformer of **Guijarro-Ordonez, Pelger & Zanotti**
(arXiv:2106.04028), which processes each stock's 30-day residual path
**independently** -- the only cross-sectional interaction in the entire model is
the final L1 normalisation.

Two additions, both tested against a faithful reimplementation of their model
(verified at 769 parameters, matching theirs exactly):

1. **Positional encoding** -- theirs has none; its temporal transformer sees an
   unordered set, with ordering supplied only by the causal CNN beneath.
2. **ISAB cross-sectional block** (Set Transformer, Lee et al. 2019) -- pool each
   day's cross-section into M learned inducing points and broadcast back.
   O(N*M), permutation-equivariant, masks a variable active universe.

**Judge on net-of-cost Sharpe and turnover, not gross.** We established on the
baseline that its Sharpe edge is volatility reduction rather than return
(mean 8.06%/yr vs 10.90%/yr for 1-day reversal, at half the vol), and that costs
are paid out of the mean. A model with higher gross and higher turnover is worse.

Reference numbers on PCA-5, 2002-2016, residual space, same blocks:

| arm | gross | turn | b/e | 2nd-half net@1bp |
|---|---|---|---|---|
| CNN+Transformer (theirs) | 4.92 | 1.00 | 3.2bp | +0.21 |
| reversal L=30 + band | 1.05 | 0.21 | 6.1bp | +0.47 |
| permute null | 0.46 | - | - | - |


## Measured results (T4)

| arm | params | gross | turn | b/e | net@1bp | 2H gross | 2H net@1bp |
|---|---|---|---|---|---|---|---|
| baseline (= their model) | 769 | **+4.92** | 1.00 | 3.2bp | +3.39 | +2.23 | +0.21 |
| **+ cross-sectional attention** | 1,785 | **+4.77** | 1.00 | 3.1bp | +3.24 | +2.25 | +0.19 |
| reversal L=30 + band | 0 | +1.05 | 0.21 | 6.1bp | +0.84 | +0.67 | **+0.47** |

**Null result.** Every column within noise (SE 0.26 full, 0.37 second half),
including the second half where cross-sectional context should have helped most.

The baseline arm reproduces their model to ~0.02 Sharpe per block, which is what
makes the comparison attributable to the ISAB block rather than to this
reimplementation.

**Why it plausibly does nothing:** residualisation has already removed the
cross-sectional structure the attention block is looking for. That predicts it
*should* help on K=0 residuals (raw returns) — untested. Consistent with the
successor paper ([arXiv:2510.11616](https://arxiv.org/abs/2510.11616)), which
puts attention at the factor-construction stage instead and reports 2.3 net.


In [ ]:
#@title Setup, GPU, and a trimmed clone of their repo (~40MB, not 840MB)
import torch, os, time, subprocess, urllib.request
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE -- Runtime > Change runtime type > GPU")

def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).returncode

if not os.path.isdir("dlsa-public"):
    sh("git clone --depth 1 --filter=blob:none --no-checkout -q "
       "https://github.com/gregzanotti/dlsa-public.git")
    sh("cd dlsa-public && git sparse-checkout set --no-cone '/*' '!/residuals/*' "
       "&& git checkout -q main")
os.chdir("dlsa-public") if os.path.basename(os.getcwd()) != "dlsa-public" else None

os.makedirs("residuals/pca", exist_ok=True)
B = "https://raw.githubusercontent.com/gregzanotti/dlsa-public/main/residuals"
F = "AvPCA_OOSresiduals_5_factors_1998_initialOOSYear_60_rollingWindow_252_covWindow_0.01_Cap.npy.gz"
for src, dst in ((f"{B}/superMask.npy", "residuals/superMask.npy"),
                 (f"{B}/pca/{F}",       f"residuals/pca/{F}")):
    if not os.path.exists(dst):
        print("downloading", os.path.basename(dst), flush=True)
        urllib.request.urlretrieve(src, dst)
print("files ready:", sorted(os.listdir("residuals")))


In [ ]:
#@title Parameters
K            = 5     #@param {type:"integer"}
LOOKBACK     = 30    #@param {type:"integer"}
EPOCHS       = 30    #@param {type:"integer"}   # their config: 100
RETRAIN_FREQ = 250   #@param {type:"integer"}   # their config: 125
TRAIN_LEN    = 1000  #@param {type:"integer"}
BATCH_DAYS   = 125   #@param {type:"integer"}
LR           = 0.001
INDUCING     = 16    #@param {type:"integer"}   # M, cross-sectional inducing points
COST_BP      = 0.0   #@param {type:"number"}    # >0 puts a friction penalty IN the objective
ARMS         = ["pos"]        # baseline 4.92 and xsection 4.77 already measured
                              # options: baseline, pos, xsection, full
                              # options: baseline, pos, xsection, full

In [ ]:
#@title Load their residuals
import gzip, shutil, numpy as np
p = f"residuals/pca/AvPCA_OOSresiduals_{K}_factors_1998_initialOOSYear_60_rollingWindow_252_covWindow_0.01_Cap.npy"
if not os.path.exists(p):
    with gzip.open(p+".gz","rb") as fi, open(p,"wb") as fo: shutil.copyfileobj(fi, fo)
resid = np.load(p); mask = np.load("residuals/superMask.npy")
data = np.asarray(resid[:, mask], dtype=np.float32); del resid
print(f"PCA-{K} residuals {data.shape}   active/day median {np.median((data!=0).sum(1)):.0f}")

In [ ]:
%%writefile xsection_model.py
"""
Cross-sectional extension of Guijarro-Ordonez / Pelger / Zanotti (2106.04028).

Their CNNTransformer processes each stock's 30-day residual path INDEPENDENTLY.
The only cross-sectional interaction in the whole model is the final L1
normalisation.  A stat-arb book is fundamentally relative, so this adds:

  1. sinusoidal positional encoding    (theirs has none at all -- the temporal
                                        transformer sees an unordered set, with
                                        ordering supplied only by the causal CNN)
  2. an ISAB cross-sectional block     (Set Transformer, Lee et al. 2019):
                                        pool the day's cross-section into M
                                        learned inducing points, then broadcast
                                        back.  O(N*M) not O(N^2), permutation-
                                        equivariant, handles a variable number
                                        of active names per day via masking.

Everything else -- CNN block, temporal encoder, Sharpe objective, rolling
retrain -- is kept identical so the comparison is clean.
"""
import math
import torch
import torch.nn as nn


class CNN_Block(nn.Module):
    """Verbatim from their models/CNNTransformer.py."""
    def __init__(self, in_filters=1, out_filters=8, normalization=True, filter_size=2):
        super().__init__()
        self.in_filters, self.out_filters = in_filters, out_filters
        self.conv1 = nn.Conv1d(in_filters, out_filters, filter_size)
        self.conv2 = nn.Conv1d(out_filters, out_filters, filter_size)
        self.relu = nn.ReLU(inplace=True)
        self.pad = nn.ConstantPad1d((filter_size-1, 0), 0)      # causal
        self.n1, self.n2 = nn.InstanceNorm1d(in_filters), nn.InstanceNorm1d(out_filters)
        self.normalization = normalization

    def forward(self, x):
        if self.normalization: x = self.n1(x)
        out = self.relu(self.conv1(self.pad(x)))
        if self.normalization: out = self.n2(out)
        out = self.relu(self.conv2(self.pad(out)))
        return out + x.repeat(1, self.out_filters // self.in_filters, 1)


class ISAB(nn.Module):
    """Induced Set Attention Block: pool the cross-section through M inducing
    points, then broadcast back.  Cost O(N*M) instead of O(N^2)."""
    def __init__(self, dim, heads=4, m=16, dropout=0.1):
        super().__init__()
        self.I = nn.Parameter(torch.randn(1, m, dim) * dim**-0.5)
        self.pool = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.cast = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, 2*dim), nn.ReLU(), nn.Linear(2*dim, dim))

    def forward(self, E, pad_mask):
        """E (D, N, C) -- D days, N stocks, C features.  pad_mask (D, N) True = inactive."""
        D = E.shape[0]
        H, _ = self.pool(self.I.expand(D, -1, -1), E, E, key_padding_mask=pad_mask)
        A, _ = self.cast(E, H, H)
        E = self.n1(E + A)
        return self.n2(E + self.ff(E))


PE_BASE = 100.0   # NOT the usual 10000: that schedule is designed for sequences
                  # of ~1000+ steps.  Over T=30, base 10000 leaves 3 of 8 dims
                  # with std < 0.05 (frequencies too slow to move).  Measured
                  # min pairwise distance between positions / dead dims:
                  #   base 10000 -> 0.658 / 3 of 8
                  #   base  1000 -> 0.976 / 2 of 8
                  #   base   100 -> 1.015 / 0 of 8   <- best separability, no waste

def positional_encoding(T, C, device, base=PE_BASE):
    pe = torch.zeros(T, C, device=device)
    pos = torch.arange(T, device=device, dtype=torch.float).unsqueeze(1)
    div = torch.exp(torch.arange(0, C, 2, device=device).float() * (-math.log(base)/C))
    pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div[:pe[:, 1::2].shape[1]])
    return pe


class XSectionNet(nn.Module):
    def __init__(self, lookback=30, filters=8, heads=4, hidden_factor=2,
                 dropout=0.25, filter_size=2, inducing=16, use_pos=True,
                 use_xsection=True, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.lookback, self.C = lookback, filters
        self.use_pos, self.use_xsection = use_pos, use_xsection
        self.cnn = CNN_Block(1, filters, True, filter_size)
        self.temporal = nn.TransformerEncoderLayer(
            d_model=filters, nhead=heads, dim_feedforward=hidden_factor*filters,
            dropout=dropout, batch_first=False)
        self.pe_scale = nn.Parameter(torch.ones(1)) if use_pos else None
        # CNN features measure |h| ~ 0.80 while the encoding has amplitude 1.0,
        # so a plain add is an arbitrary 1.3x weighting.  Let the model choose.
        # Init 1.0 => identical to standard PE at initialisation.
        self.pe_scale = nn.Parameter(torch.ones(1)) if use_pos else None
        self.xsec = ISAB(filters, heads, inducing, dropout) if use_xsection else None
        self.out = nn.Linear(filters, 1)

    MAX_ROWS = 32768   # their attention kernel caps batch at 65535 rows

    def _embed_chunk(self, x):
        n, T = x.shape
        h = self.cnn(x.reshape(n, 1, T))            # (n, C, T)
        h = h.permute(2, 0, 1)                      # (T, n, C)
        if self.use_pos:
            h = h + self.pe_scale * positional_encoding(T, self.C, x.device).unsqueeze(1)
        h = self.temporal(h)
        return h[-1]                                # (n, C)

    def embed(self, x):
        """x (n, T) flat (day,stock) pairs -> (n, C).  Chunked; gradients flow
        through the cat, so this is identical to one pass."""
        if x.shape[0] <= self.MAX_ROWS:
            return self._embed_chunk(x)
        return torch.cat([self._embed_chunk(x[i:i+self.MAX_ROWS])
                          for i in range(0, x.shape[0], self.MAX_ROWS)])

    def forward(self, x, sub=None):
        """x (n, T) flat pairs.  sub (D, N) bool placing each pair in its day.
        Without sub, behaves like their per-stock model."""
        e = self.embed(x)
        if self.xsec is None or sub is None:
            return self.out(e).squeeze(-1)
        D, N = sub.shape
        E = torch.zeros(D, N, self.C, device=x.device, dtype=e.dtype)
        E[sub] = e
        E = self.xsec(E, pad_mask=~sub)             # stocks see each other
        return self.out(E[sub]).squeeze(-1)


In [ ]:
#@title Their preprocessing, our model
import sys; sys.path.insert(0, ".")
from preprocess import preprocess_cumsum          # THEIR code
from xsection_model import XSectionNet            # ours
windows, idxs = preprocess_cumsum(data, LOOKBACK)
idxs = idxs.numpy() if hasattr(idxs, "numpy") else idxs
print("windows", windows.shape, " valid (day,stock) pairs", idxs.sum())

In [ ]:
#@title Training loop -- identical protocol, Sharpe objective, turnover tracked
import json
CFG = {"baseline": dict(use_pos=False, use_xsection=False),
       "pos":      dict(use_pos=True,  use_xsection=False),
       "xsection": dict(use_pos=False, use_xsection=True),
       "full":     dict(use_pos=True,  use_xsection=True)}

def sharpe(x): return float(x.mean()/x.std()*np.sqrt(252)) if len(x) > 20 else float("nan")

def train_eval(arm, seed=0):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed); np.random.seed(seed)
    Tn, N = windows.shape[0], idxs.shape[1]
    blocks, all_r, all_to = [], [], []
    prev_w = np.zeros(N)
    starts = list(range(TRAIN_LEN, Tn - RETRAIN_FREQ, RETRAIN_FREQ))
    for bi, s0 in enumerate(starts):
        model = XSectionNet(lookback=LOOKBACK, inducing=INDUCING, seed=seed, **CFG[arm]).to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        tr = np.arange(s0-TRAIN_LEN, s0)
        for ep in range(EPOCHS):
            for ch in [tr[i:i+BATCH_DAYS] for i in range(0, len(tr), BATCH_DAYS)]:
                sub = idxs[ch]
                x = torch.tensor(windows[ch][sub], device=dev)
                if len(x) < 10: continue
                st = torch.tensor(sub, device=dev)
                W = torch.zeros(len(ch), N, device=dev)
                W[st] = model(x, st)
                W = W / W.abs().sum(1, keepdim=True).clamp(min=1e-9)
                r = (W * torch.tensor(data[LOOKBACK+ch], device=dev)).sum(1)
                if COST_BP > 0:                      # friction penalty in the objective
                    r = r - COST_BP*1e-4*(W[1:]-W[:-1]).abs().sum(1).mean()
                loss = -r.mean()/r.std().clamp(min=1e-9)
                opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); br, bt = [], []
        te = np.arange(s0, min(s0+RETRAIN_FREQ, Tn))
        with torch.no_grad():
            for ch in [te[i:i+BATCH_DAYS] for i in range(0, len(te), BATCH_DAYS)]:
                sub = idxs[ch]
                x = torch.tensor(windows[ch][sub], device=dev)
                if len(x) < 10: continue
                st = torch.tensor(sub, device=dev)
                W = torch.zeros(len(ch), N, device=dev)
                W[st] = model(x, st)
                W = W / W.abs().sum(1, keepdim=True).clamp(min=1e-9)
                br.append((W*torch.tensor(data[LOOKBACK+ch], device=dev)).sum(1).cpu().numpy())
                Wc = W.cpu().numpy().astype(np.float64)
                for k in range(len(Wc)): bt.append(np.abs(Wc[k]-prev_w).sum()); prev_w = Wc[k]
        br = np.concatenate(br) if br else np.array([]); bt = np.array(bt)
        all_r.append(br); all_to.append(bt)
        if len(br) > 20:
            blocks.append({"year": round(1998.0+s0/252.0,2), "sharpe": sharpe(br),
                           "turnover": float(bt.mean())})
            # checkpoint EVERY block -- Colab has been dropping the runtime
            json.dump({"arm": arm, "blocks": blocks}, open(f"ckpt_{arm}.json","w"), indent=1)
            np.save(f"partial_{arm}.npy",
                    np.vstack([np.concatenate([a for a in all_r if len(a)]),
                               np.concatenate([a for a in all_to if len(a)])]))
            print(f"    block {bi+1}/{len(starts)}  ~{blocks[-1]['year']:.1f}  "
                  f"SR {blocks[-1]['sharpe']:+.2f}  turn {blocks[-1]['turnover']:.2f}"
                  f"   ({time.time()-T0:.0f}s)", flush=True)
    r = np.concatenate([a for a in all_r if len(a)]); to = np.concatenate([a for a in all_to if len(a)])
    return r, to, blocks

def be(g, t):
    lo, hi = 0.0, 500.0
    for _ in range(60):
        m = (lo+hi)/2
        if sharpe(g - m*1e-4*t) > 0: lo = m
        else: hi = m
    return lo

In [ ]:
#@title Run
T0 = time.time(); out = {}
for arm in ARMS:
    print(f"\n=== {arm} ===", flush=True)
    r, to, blocks = train_eval(arm)
    h = len(r)//2
    out[arm] = {"gross": sharpe(r), "turnover": float(to.mean()),
                "breakeven_bp": be(r, to), "net_1bp": sharpe(r-1e-4*to),
                "second_half_gross": sharpe(r[h:]),
                "second_half_net_1bp": sharpe(r[h:]-1e-4*to[h:]),
                "second_half_be": be(r[h:], to[h:]),
                "blocks": blocks}
    np.save(f"w_{arm}.npy", np.vstack([r, to]))
    json.dump(out, open("xsection_results.json","w"), indent=2)
    print(f"  gross {out[arm]['gross']:+.2f}  turn {out[arm]['turnover']:.2f}  "
          f"b/e {out[arm]['breakeven_bp']:.1f}bp  net@1bp {out[arm]['net_1bp']:+.2f}")

print("\n"+"="*84)
print(f"{'arm':22s} {'gross':>7s} {'turn':>6s} {'b/e bp':>7s} {'net@1bp':>8s} {'2H gross':>9s} {'2H net@1bp':>11s}")
print("-"*84)
for a, v in out.items():
    print(f"{a:22s} {v['gross']:+7.2f} {v['turnover']:6.2f} {v['breakeven_bp']:7.1f} "
          f"{v['net_1bp']:+8.2f} {v['second_half_gross']:+9.2f} {v['second_half_net_1bp']:+11.2f}")
print(f"{'-- xsection (measured)':22s} {4.77:+7.2f} {1.00:6.2f} {3.1:7.1f} {3.24:+8.2f} {2.25:+9.2f} {0.19:+11.2f}")
print(f"{'-- baseline (measured)':22s} {4.92:+7.2f} {1.00:6.2f} {3.2:7.1f} {3.39:+8.2f} {2.23:+9.2f} {0.21:+11.2f}")
print(f"{'-- reversal L=30+band':22s} {1.05:+7.2f} {0.21:6.2f} {6.1:7.1f} {0.84:+8.2f} {0.67:+9.2f} {0.47:+11.2f}")
print("="*84)
print("SE(Sharpe) ~ 0.26 full, 0.37 second half -- treat sub-0.4 gaps as noise.")

from google.colab import files
for f in ["xsection_results.json"] + [f"w_{a}.npy" for a in ARMS]:
    try: files.download(f)
    except Exception as e: print("grab", f, "manually:", e)